# Сборка Android APK в Google Colab

Этот notebook автоматически соберет Android APK из проекта.

## Инструкция:
1. Запустите все ячейки по порядку (Runtime → Run all)
2. Первая сборка займет 30-60 минут (скачивание SDK/NDK)
3. Готовый APK можно будет скачать в последней ячейке

## Шаг 1: Установка зависимостей

Устанавливаем системные пакеты и Python библиотеки.

In [ ]:
%%bash
# Обновляем систему
apt-get update -qq

# Устанавливаем системные зависимости
apt-get install -y -qq \
    openjdk-17-jdk \
    build-essential \
    libssl-dev \
    libffi-dev \
    python3-dev \
    libtool \
    automake \
    autoconf \
    libltdl-dev \
    pkg-config \
    m4 \
    git \
    unzip \
    wget

echo "✓ Системные зависимости установлены"

# Устанавливаем Python зависимости
pip3 install --quiet --user buildozer cython>=0.29.21

# Добавляем в PATH
export PATH="$HOME/.local/bin:$PATH"

echo "✓ Python зависимости установлены"
echo "✓ Buildozer версия:"
buildozer --version

## Шаг 2: Загрузка проекта

Выберите один из вариантов:
- **Вариант A**: Клонирование из GitHub (если проект на GitHub)
- **Вариант B**: Загрузка файлов вручную через Colab UI

In [ ]:
# ВАРИАНТ A: Клонирование из GitHub
# Раскомментируйте и укажите URL вашего репозитория:

# !git clone https://github.com/your-username/your-repo.git
# %cd your-repo/ready_app

# Или используйте этот репозиторий:
!git clone https://github.com/suninvest777/mallware.git || echo "Репозиторий уже склонирован"
%cd mallware/ready_app

import os
print("✓ Проект загружен")
print(f"Текущая директория: {os.getcwd()}")
!ls -la

In [ ]:
# ВАРИАНТ B: Загрузка файлов вручную
# Если вы загрузили файлы через Colab UI (Files → Upload),
# раскомментируйте и укажите путь:

# import os
# os.chdir('/content/your-uploaded-folder/ready_app')
# print(f"Текущая директория: {os.getcwd()}")
# !ls -la

## Шаг 3: Применение патча для pyjnius

Патч исправляет проблемы совместимости Python 2/3 в pyjnius.

In [ ]:
%%bash
# Проверяем наличие патч-скрипта
if [ -f "patch_pyjnius.sh" ]; then
  echo "Применяем патч для pyjnius..."
  chmod +x patch_pyjnius.sh
  bash patch_pyjnius.sh || echo "Патч будет применен позже (pyjnius еще не установлен)"
else
  echo "⚠ Файл patch_pyjnius.sh не найден, создаем его..."
  cat > patch_pyjnius.sh << 'EOF'
#!/bin/bash
echo "=== Применение патча для pyjnius ==="
PYJNIUS_PATH=$(find .buildozer -path "*/pyjnius/jnius/jnius_utils.pxi" 2>/dev/null | head -1)
if [ -z "$PYJNIUS_PATH" ]; then
  PYJNIUS_PATH=$(find $HOME/.buildozer -path "*/pyjnius/jnius/jnius_utils.pxi" 2>/dev/null | head -1)
fi
if [ -n "$PYJNIUS_PATH" ] && [ -f "$PYJNIUS_PATH" ]; then
  echo "Найден файл: $PYJNIUS_PATH"
  cp "$PYJNIUS_PATH" "$PYJNIUS_PATH.bak"
  sed -i 's/isinstance(arg, long)/isinstance(arg, int)/g' "$PYJNIUS_PATH"
  sed -i 's/ isinstance(arg, long)/ isinstance(arg, int)/g' "$PYJNIUS_PATH"
  sed -i 's/(isinstance(arg, long)/(isinstance(arg, int)/g' "$PYJNIUS_PATH"
  sed -i 's/( isinstance(arg, long)/( isinstance(arg, int)/g' "$PYJNIUS_PATH"
  sed -i 's/(arg, long)/(arg, int)/g' "$PYJNIUS_PATH"
  sed -i 's/( arg, long)/( arg, int)/g' "$PYJNIUS_PATH"
  sed -i 's/isinstance([^,]*,\s*long\b)/isinstance(\1, int)/g' "$PYJNIUS_PATH"
  echo "✓ Патч применен"
else
  echo "⚠ Файл jnius_utils.pxi не найден, патч будет применен позже"
fi
EOF
  chmod +x patch_pyjnius.sh
  echo "✓ Патч-скрипт создан"
fi

## Шаг 4: Инициализация Buildozer и Android SDK

Устанавливаем Android SDK, NDK и принимаем лицензии.

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"

# Создаем структуру SDK и принимаем лицензии
SDK_PATH="$HOME/.buildozer/android/platform/android-sdk"
mkdir -p "$SDK_PATH/licenses"

# Принимаем все лицензии автоматически
echo "24333f8a63b6825ea9c5514f83c2829b004d1fee" > "$SDK_PATH/licenses/android-sdk-license"
echo "84831b9409646a918e30573bab4c9c91346d8abd" > "$SDK_PATH/licenses/android-sdk-preview-license"
echo "8403addf88ab4874007e1c1e80a0025de2550a16" > "$SDK_PATH/licenses/android-googletv-license"
echo "601085b94cd77f0b54ff86406957099ebe79c4d6" > "$SDK_PATH/licenses/android-sdk-arm-dbt-license"
echo "33b6a2b64607f11b759f320ef9dff4ae5c47d97a" > "$SDK_PATH/licenses/google-gdk-license"
echo "d975f751698a77b662f1254ddbeed3901e976f5a" > "$SDK_PATH/licenses/intel-android-extra-license"
echo "e9acab5e5f1efb7a6c54c6850d6c4c78afd05907" > "$SDK_PATH/licenses/mips-android-sysimage-license"

echo "✓ Лицензии приняты"

# Инициализируем buildozer (скачает Android SDK автоматически)
echo "Инициализация Buildozer (это может занять несколько минут)..."
buildozer android update 2>&1 | tail -20 || echo "Buildozer update завершен"

echo "✓ Buildozer инициализирован"

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
SDK_PATH="$HOME/.buildozer/android/platform/android-sdk"

# Устанавливаем cmdline-tools если нужно
if [ -d "$SDK_PATH" ] && [ ! -d "$SDK_PATH/cmdline-tools/latest" ]; then
  echo "Устанавливаем cmdline-tools..."
  cd /tmp
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip || \
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-9477386_latest.zip
  
  if [ -f commandlinetools-*.zip ]; then
    TEMP_DIR=$(mktemp -d)
    unzip -q commandlinetools-*.zip -d "$TEMP_DIR"
    mkdir -p "$SDK_PATH/cmdline-tools/latest"
    
    if [ -d "$TEMP_DIR/cmdline-tools" ]; then
      mv "$TEMP_DIR/cmdline-tools"/* "$SDK_PATH/cmdline-tools/latest/" 2>/dev/null || true
    else
      mv "$TEMP_DIR"/* "$SDK_PATH/cmdline-tools/latest/" 2>/dev/null || true
    fi
    
    rm -rf "$TEMP_DIR"
    rm -f commandlinetools-*.zip
    echo "✓ cmdline-tools установлены"
  fi
fi

# Создаем симлинк для sdkmanager
if [ -d "$SDK_PATH" ]; then
  mkdir -p "$SDK_PATH/tools/bin"
  
  if [ -f "$SDK_PATH/cmdline-tools/latest/bin/sdkmanager" ]; then
    ln -sf "$SDK_PATH/cmdline-tools/latest/bin/sdkmanager" "$SDK_PATH/tools/bin/sdkmanager"
    echo "✓ Симлинк для sdkmanager создан"
  fi
  
  # Устанавливаем build-tools и платформу
  SDKMANAGER=""
  if [ -f "$SDK_PATH/tools/bin/sdkmanager" ]; then
    SDKMANAGER="$SDK_PATH/tools/bin/sdkmanager"
  elif [ -f "$SDK_PATH/cmdline-tools/latest/bin/sdkmanager" ]; then
    SDKMANAGER="$SDK_PATH/cmdline-tools/latest/bin/sdkmanager"
  fi
  
  if [ -n "$SDKMANAGER" ]; then
    echo "Установка build-tools и платформы android-33..."
    $SDKMANAGER "build-tools;34.0.0" "platform-tools" "platforms;android-33" 2>&1 | tail -10 || \
    $SDKMANAGER "build-tools;33.0.0" "platform-tools" "platforms;android-33" 2>&1 | tail -10 || \
    $SDKMANAGER "build-tools;32.0.0" "platform-tools" "platforms;android-33" 2>&1 | tail -10
    echo "✓ Android SDK компоненты установлены"
  fi
fi

## Шаг 5: Применение патча и проверка .c файлов

Применяем патч для pyjnius и проверяем генерацию .c файлов Cython.

In [ ]:
%%bash
# Применяем патч для pyjnius
if [ -f "patch_pyjnius.sh" ]; then
  echo "Применение патча для pyjnius..."
  bash patch_pyjnius.sh || echo "Патч будет применен после установки pyjnius"
fi

# Проверяем и генерируем .c файлы Cython для pyjnius
PYJNIUS_DIR=$(find .buildozer -type d -name "pyjnius" 2>/dev/null | head -1)
if [ -z "$PYJNIUS_DIR" ]; then
  PYJNIUS_DIR=$(find $HOME/.buildozer -type d -name "pyjnius" 2>/dev/null | head -1)
fi

if [ -n "$PYJNIUS_DIR" ] && [ -d "$PYJNIUS_DIR" ]; then
  echo "Найден pyjnius в: $PYJNIUS_DIR"
  JNIUS_PYX="$PYJNIUS_DIR/jnius/jnius.pyx"
  JNIUS_C="$PYJNIUS_DIR/jnius/jnius.c"
  
  if [ -f "$JNIUS_PYX" ]; then
    if [ ! -f "$JNIUS_C" ]; then
      echo "Генерируем .c файл из .pyx..."
      cd "$PYJNIUS_DIR/jnius"
      cython jnius.pyx 2>&1 || python3 -m Cython jnius.pyx 2>&1 || echo "Не удалось сгенерировать .c файл"
      cd - >/dev/null
    fi
    
    if [ -f "$JNIUS_C" ]; then
      echo "✓ Файл jnius.c найден"
    else
      echo "⚠ Файл jnius.c не найден, но продолжаем"
    fi
  fi
else
  echo "⚠ pyjnius еще не установлен, будет установлен во время сборки"
fi

## Шаг 6: Сборка APK

**ВНИМАНИЕ**: Первая сборка займет 30-60 минут!

Buildozer скачает Android NDK, скомпилирует Python и все зависимости.

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
export ACLOCAL_PATH="/usr/share/aclocal:$ACLOCAL_PATH"

echo "Начинаем сборку APK..."
echo "Это может занять 30-60 минут при первой сборке"
echo ""

# Применяем патч перед сборкой (на случай если pyjnius установился)
if [ -f "patch_pyjnius.sh" ]; then
  bash patch_pyjnius.sh 2>&1 | grep -E "(Патч|найден|применен)" || true
fi

# Запускаем сборку
buildozer android debug 2>&1 | tee /tmp/buildozer.log

BUILD_RESULT=${PIPESTATUS[0]}

if [ $BUILD_RESULT -eq 0 ]; then
  echo ""
  echo "✓✓✓ СБОРКА УСПЕШНА! ✓✓✓"
else
  echo ""
  echo "✗✗✗ ОШИБКА СБОРКИ ✗✗✗"
  echo "Последние строки лога:"
  tail -50 /tmp/buildozer.log
fi

## Шаг 7: Скачивание APK

Находим собранный APK и предоставляем ссылку для скачивания.

In [ ]:
import os
import glob
from google.colab import files

# Ищем APK файл
apk_patterns = [
    "bin/*.apk",
    "../bin/*.apk",
    "*.apk"
]

apk_file = None
for pattern in apk_patterns:
    matches = glob.glob(pattern)
    if matches:
        apk_file = matches[0]
        break

if apk_file and os.path.exists(apk_file):
    file_size = os.path.getsize(apk_file) / (1024 * 1024)  # MB
    print(f"✓ APK найден: {apk_file}")
    print(f"  Размер: {file_size:.2f} MB")
    print(f"  Полный путь: {os.path.abspath(apk_file)}")
    print("")
    print("Скачивание APK...")
    files.download(apk_file)
    print("✓ APK скачан!")
else:
    print("✗ APK файл не найден")
    print("Проверяем содержимое bin/:")
    if os.path.exists("bin"):
        print(os.listdir("bin"))
    else:
        print("Папка bin/ не существует")
    print("")
    print("Возможные причины:")
    print("- Сборка еще не завершена")
    print("- Произошла ошибка во время сборки")
    print("- APK находится в другом месте")

## Устранение неполадок

### Если сборка не удалась:

1. Проверьте логи в ячейке сборки
2. Убедитесь, что все зависимости установлены
3. Попробуйте перезапустить runtime (Runtime → Restart runtime)
4. Проверьте, что файлы проекта загружены правильно

### Если APK не найден:

1. Проверьте папку `bin/` вручную
2. Убедитесь, что сборка завершилась успешно
3. Проверьте логи на наличие ошибок

### Полезные команды:

```bash
# Проверить версию buildozer
!buildozer --version

# Посмотреть содержимое bin/
!ls -la bin/

# Проверить логи сборки
!tail -100 /tmp/buildozer.log
```